# 从零实现 PPO Actor-Critic：GAE、裁剪目标与策略发布

本 Notebook 不调用强化学习框架，也不把 PPO 简化成一行库函数。我们从一个可审计的离散环境开始，手写 `ActorCritic52(nn.Module)`、rollout、GAE、clipped surrogate、value/entropy loss、训练与确定性评估，并把 `terminated` 与 `truncated` 的 bootstrap 语义分开。

目标不是用四步合成环境证明 PPO 能解决真实控制，而是建立一组会杀死常见错误实现的 oracle：旧策略必须 detach、负优势的裁剪方向不能写反、时间截断是否 bootstrap 必须进入发布合同，模型、环境、动作语义和训练 recipe 不能靠制品内部“自签哈希”冒充可信。

In [ ]:
import copy
import hashlib
import io
import json
import math
import random
import warnings
from types import MappingProxyType

warnings.filterwarnings("ignore", message="The pynvml package is deprecated")
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

SEED52 = 5201
random.seed(SEED52); np.random.seed(SEED52); torch.manual_seed(SEED52)
torch.set_num_threads(1)
DEVICE52 = torch.device("cpu")

def canonical_json52(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))

def sha256_bytes52(value):
    return hashlib.sha256(value).hexdigest()

assert DEVICE52.type == "cpu" and torch.get_num_threads() == 1
assert torch.initial_seed() == SEED52

## 1. 环境、状态与结束语义

环境有四个阶段，状态是 `[state_dim=4]` 的 one-hot，动作空间是 `{0,1}`，正确动作序列为 `[0,1,1,0]`。正确奖励 `+1`，错误奖励 `-0.25`。完成第四步是 MDP 的 `terminated`；人为缩短 `max_steps` 才是 `truncated`。

这一区分直接影响 TD target。终止状态没有未来回报；时间上限通常仍有一个未观测的后继价值，是否 bootstrap 是算法合同而不是布尔变量命名习惯。

In [ ]:
class PatternEnv52:
    def __init__(self, target=(0, 1, 1, 0), max_steps=None):
        if not target or any(a not in (0, 1) for a in target):
            raise ValueError("target_must_be_nonempty_binary_sequence")
        self.target = tuple(int(a) for a in target)
        self.state_dim = len(self.target)
        self.max_steps = self.state_dim if max_steps is None else int(max_steps)
        if not 1 <= self.max_steps <= self.state_dim:
            raise ValueError("invalid_max_steps")
        self.index = 0
        self.done = False

    def _state(self):
        state = torch.zeros(self.state_dim, dtype=torch.float32)
        if self.index < self.state_dim:
            state[self.index] = 1.0
        return state

    def reset(self):
        self.index = 0; self.done = False
        return self._state()

    def step(self, action):
        if self.done:
            raise RuntimeError("episode_already_finished")
        if isinstance(action, bool) or not isinstance(action, (int, np.integer)) or int(action) not in (0, 1):
            raise ValueError("action_must_be_integer_0_or_1")
        correct = int(action) == self.target[self.index]
        reward = 1.0 if correct else -0.25
        self.index += 1
        terminated = self.index == self.state_dim
        truncated = self.index == self.max_steps and not terminated
        self.done = terminated or truncated
        return self._state(), reward, terminated, truncated, {"correct": correct, "stage": self.index - 1}

env52 = PatternEnv52()
state52 = env52.reset()
assert state52.shape == (4,) and state52.argmax().item() == 0
_, reward52, term52, trunc52, info52 = env52.step(0)
assert reward52 == 1.0 and info52["correct"] and not term52 and not trunc52
short52 = PatternEnv52(max_steps=2); short52.reset(); short52.step(0)
next52, _, term52, trunc52, _ = short52.step(1)
assert trunc52 and not term52 and next52.argmax().item() == 2
try:
    short52.step(0); raise AssertionError("done episode accepted another step")
except RuntimeError as error:
    assert str(error) == "episode_already_finished"

## 2. 手写共享干路 Actor-Critic

对 batch 状态 `x:[B,4]`，共享干路产生 `h:[B,H]`；actor 输出 `logits:[B,2]`，critic 输出 `V(s):[B]`。策略为 $\pi(a|s)=\mathrm{softmax}(logits)_a$。

rollout 采样使用显式 `torch.Generator`，便于回归测试。策略动作的 log-probability 与 value 在写入 buffer 时立即 detach；PPO 更新期间只允许新策略一侧建立梯度图。

In [ ]:
class ActorCritic52(nn.Module):
    def __init__(self, state_dim=4, hidden_dim=24, action_dim=2):
        super().__init__()
        if state_dim < 1 or hidden_dim < 2 or action_dim < 2:
            raise ValueError("invalid_actor_critic_config")
        self.state_dim, self.hidden_dim, self.action_dim = state_dim, hidden_dim, action_dim
        self.trunk = nn.Sequential(nn.Linear(state_dim, hidden_dim), nn.Tanh())
        self.actor = nn.Linear(hidden_dim, action_dim)
        self.critic = nn.Linear(hidden_dim, 1)

    def forward(self, states):
        if states.ndim != 2 or states.shape[1] != self.state_dim:
            raise ValueError("states_must_have_shape_B_state_dim")
        if not torch.isfinite(states).all():
            raise ValueError("nonfinite_states")
        if not torch.all((states == 0) | (states == 1)) or not torch.allclose(states.sum(-1), torch.ones(states.shape[0], device=states.device)):
            raise ValueError("state_encoding_must_be_one_hot")
        hidden = self.trunk(states)
        return self.actor(hidden), self.critic(hidden).squeeze(-1)

    @torch.no_grad()
    def act(self, states, generator, deterministic=False):
        logits, values = self(states)
        probs = logits.softmax(-1)
        actions = probs.argmax(-1) if deterministic else torch.multinomial(probs, 1, generator=generator).squeeze(1)
        log_probs = probs.gather(1, actions[:, None]).squeeze(1).log()
        return actions, log_probs, values

probe_model52 = ActorCritic52()
probe_states52 = torch.eye(4)
logits52, values52 = probe_model52(probe_states52)
assert logits52.shape == (4, 2) and values52.shape == (4,)
assert torch.allclose(logits52.softmax(-1).sum(-1), torch.ones(4), atol=1e-7)
g_a52 = torch.Generator().manual_seed(7); g_b52 = torch.Generator().manual_seed(7)
assert torch.equal(probe_model52.act(probe_states52, g_a52)[0], probe_model52.act(probe_states52, g_b52)[0])
try:
    probe_model52(torch.tensor([[float("nan"), 0, 0, 0]])); raise AssertionError("NaN state accepted")
except ValueError as error:
    assert str(error) == "nonfinite_states"
try:
    probe_model52(torch.zeros(1,4)); raise AssertionError("all-zero state accepted")
except ValueError as error:
    assert str(error) == "state_encoding_must_be_one_hot"

## 3. GAE：bootstrap mask 与递推边界不是一回事

$$\delta_t=r_t+\gamma m_t^{boot}V(s_{t+1})-V(s_t),$$
$$A_t=\delta_t+\gamma\lambda m_t^{continue}A_{t+1}.$$

`terminated` 总令两个 mask 为 0。若合同允许 time-limit bootstrap，最后一个 `truncated` transition 的 `m_boot=1`，但 `m_continue=0`，否则优势会跨到下一条 episode。输入为 `rewards:[T]`、`values:[T+1]` 和两个 bool flag `[T]`。

In [ ]:
def generalized_advantage52(rewards, values, terminated, truncated, gamma=0.9, lam=0.8, bootstrap_on_truncation=True):
    if rewards.ndim != 1 or values.shape != (rewards.numel() + 1,):
        raise ValueError("gae_shape_contract")
    if not rewards.is_floating_point() or not values.is_floating_point():
        raise TypeError("gae_rewards_values_must_be_floating")
    if rewards.dtype != values.dtype or rewards.device != values.device:
        raise ValueError("gae_rewards_values_dtype_device_mismatch")
    if terminated.shape != rewards.shape or truncated.shape != rewards.shape:
        raise ValueError("gae_flag_shape_contract")
    if terminated.dtype != torch.bool or truncated.dtype != torch.bool:
        raise TypeError("gae_flags_must_be_bool")
    if bool((terminated & truncated).any()):
        raise ValueError("transition_cannot_be_terminated_and_truncated")
    if not math.isfinite(gamma) or not math.isfinite(lam) or not 0 <= gamma <= 1 or not 0 <= lam <= 1:
        raise ValueError("gamma_lambda_range_contract")
    if not torch.isfinite(rewards).all() or not torch.isfinite(values).all():
        raise ValueError("nonfinite_gae_input")
    advantages = torch.zeros_like(rewards)
    gae = torch.zeros((), dtype=rewards.dtype)
    for t in range(rewards.numel() - 1, -1, -1):
        boot = ~terminated[t]
        if not bootstrap_on_truncation:
            boot = boot & ~truncated[t]
        continuation = ~(terminated[t] | truncated[t])
        delta = rewards[t] + gamma * boot.to(rewards.dtype) * values[t + 1] - values[t]
        gae = delta + gamma * lam * continuation.to(rewards.dtype) * gae
        advantages[t] = gae
    return advantages, advantages + values[:-1]

rewards_oracle52 = torch.tensor([1.0, 1.0])
values_oracle52 = torch.tensor([0.5, 0.4, 0.3])
flags_false52 = torch.tensor([False, False]); terminal52 = torch.tensor([False, True])
adv52, returns52 = generalized_advantage52(rewards_oracle52, values_oracle52, terminal52, flags_false52)
assert torch.allclose(adv52, torch.tensor([1.292, 0.6]), atol=1e-6)
assert torch.allclose(returns52, adv52 + values_oracle52[:-1])
truncated52 = torch.tensor([False, True])
adv_boot52, _ = generalized_advantage52(rewards_oracle52, values_oracle52, flags_false52, truncated52, bootstrap_on_truncation=True)
adv_stop52, _ = generalized_advantage52(rewards_oracle52, values_oracle52, flags_false52, truncated52, bootstrap_on_truncation=False)
assert torch.isclose(adv_boot52[-1], torch.tensor(0.87), atol=1e-6)
assert torch.isclose(adv_stop52[-1], torch.tensor(0.6), atol=1e-6)
try:
    generalized_advantage52(rewards_oracle52, values_oracle52, terminal52.long(), flags_false52); raise AssertionError("integer terminal accepted")
except TypeError as error:
    assert str(error) == "gae_flags_must_be_bool"
try:
    generalized_advantage52(torch.tensor([1.]),torch.tensor([0.,0.]),torch.tensor([True]),torch.tensor([True])); raise AssertionError("contradictory flags accepted")
except ValueError as error:
    assert str(error) == "transition_cannot_be_terminated_and_truncated"
try:
    generalized_advantage52(torch.tensor([1]),torch.tensor([0,0]),torch.tensor([False]),torch.tensor([False])); raise AssertionError("integer GAE accepted")
except TypeError as error:
    assert str(error) == "gae_rewards_values_must_be_floating"

## 4. Rollout buffer 与 episode 隔离

每个 episode 单独计算 GAE，再拼接为 `[N]` batch。这样 terminal/truncation 边界不会被 batch 排列破坏。buffer 保存 `states/actions/old_log_probs/advantages/returns`；旧 log-probability 不允许 `requires_grad=True`。

生产环境还要保存 policy version、observation normalization version、action mask 和环境 build ID；否则“同一状态”并不一定有同一语义。

In [ ]:
class RolloutBatch52(dict):
    REQUIRED = ("states", "actions", "old_log_probs", "advantages", "returns")
    def __init__(self, payload):
        if tuple(payload.keys()) != self.REQUIRED:
            raise ValueError("rollout_batch_schema_mismatch")
        lengths = {tensor.shape[0] for tensor in payload.values()}
        if len(lengths) != 1 or payload["states"].ndim != 2 or any(payload[name].ndim != 1 for name in self.REQUIRED[1:]):
            raise ValueError("rollout_batch_length_mismatch")
        states, actions = payload["states"], payload["actions"]
        if states.shape[1] != 4 or not torch.all((states == 0) | (states == 1)) or not torch.allclose(states.sum(-1), torch.ones(states.shape[0])):
            raise ValueError("rollout_state_schema_mismatch")
        if actions.dtype != torch.long or bool(((actions < 0) | (actions >= 2)).any()):
            raise ValueError("rollout_action_schema_mismatch")
        if any(not torch.isfinite(tensor).all() for tensor in payload.values()):
            raise ValueError("nonfinite_rollout_batch")
        super().__init__(payload)

def collect_rollouts52(model, episodes, generator, max_steps=4):
    records = {name: [] for name in ("states", "actions", "old_log_probs", "advantages", "returns")}
    for _ in range(episodes):
        env = PatternEnv52(max_steps=max_steps)
        state = env.reset(); states=[]; actions=[]; logps=[]; rewards=[]; values=[]; terms=[]; truncs=[]
        while True:
            with torch.no_grad():
                action, logp, value = model.act(state[None, :], generator)
            next_state, reward, terminated, truncated, _ = env.step(int(action.item()))
            states.append(state); actions.append(action.squeeze(0)); logps.append(logp.squeeze(0)); values.append(value.squeeze(0))
            rewards.append(reward); terms.append(terminated); truncs.append(truncated)
            state = next_state
            if terminated or truncated:
                break
        with torch.no_grad():
            next_value = torch.zeros(()) if terms[-1] else model(state[None, :])[1].squeeze(0)
        value_path = torch.stack(values + [next_value])
        adv, ret = generalized_advantage52(torch.tensor(rewards), value_path, torch.tensor(terms), torch.tensor(truncs))
        records["states"].append(torch.stack(states)); records["actions"].append(torch.stack(actions))
        records["old_log_probs"].append(torch.stack(logps)); records["advantages"].append(adv); records["returns"].append(ret)
    batch = RolloutBatch52({name: torch.cat(parts).detach() for name, parts in records.items()})
    if batch["old_log_probs"].requires_grad:
        raise RuntimeError("old_policy_must_be_detached")
    return batch

rollout_probe52 = collect_rollouts52(probe_model52, 3, torch.Generator().manual_seed(9))
assert rollout_probe52["states"].shape == (12, 4)
assert rollout_probe52["actions"].shape == rollout_probe52["returns"].shape == (12,)
assert not any(t.requires_grad for t in rollout_probe52.values())
short_rollout52 = collect_rollouts52(probe_model52, 2, torch.Generator().manual_seed(10), max_steps=2)
assert short_rollout52["states"].shape[0] == 4
invalid_rollout52 = {name: tensor.clone() for name,tensor in rollout_probe52.items()}; invalid_rollout52["actions"][0] = -1
try:
    RolloutBatch52(invalid_rollout52); raise AssertionError("negative rollout action accepted")
except ValueError as error:
    assert str(error) == "rollout_action_schema_mismatch"

## 5. PPO clipped objective

令 $r_t(\theta)=\exp(\log\pi_\theta(a_t|s_t)-\log\pi_{old}(a_t|s_t))$，策略目标为

$$L^{clip}=\mathbb E[\min(r_tA_t,\operatorname{clip}(r_t,1-\epsilon,1+\epsilon)A_t)].$$

对负优势，`min` 会选择更负的项；把它写成对 ratio 的无条件 clamp 会改变目标。总损失再加 value MSE 并减去 entropy bonus。优势只用当前 batch 的固定统计量标准化。

In [ ]:
def clipped_surrogate52(new_logp, old_logp, advantages, clip_ratio=0.2):
    if new_logp.shape != old_logp.shape or new_logp.shape != advantages.shape:
        raise ValueError("surrogate_shape_contract")
    if old_logp.requires_grad:
        raise ValueError("old_logp_must_be_detached")
    if not math.isfinite(clip_ratio) or not 0 < clip_ratio < 1 or not torch.isfinite(new_logp).all() or not torch.isfinite(old_logp).all() or not torch.isfinite(advantages).all():
        raise ValueError("surrogate_finite_clip_contract")
    log_ratio = new_logp - old_logp
    if bool((log_ratio.abs() > 20).any()):
        raise ValueError("log_ratio_out_of_safe_range")
    ratio = log_ratio.exp()
    if not torch.isfinite(ratio).all():
        raise ValueError("nonfinite_probability_ratio")
    return torch.minimum(ratio * advantages, ratio.clamp(1 - clip_ratio, 1 + clip_ratio) * advantages), ratio

def ppo_loss52(model, batch, clip_ratio=0.2, value_coef=0.5, entropy_coef=0.02):
    if not math.isfinite(value_coef) or not math.isfinite(entropy_coef) or value_coef < 0 or entropy_coef < 0:
        raise ValueError("ppo_loss_coefficient_contract")
    logits, values = model(batch["states"])
    log_all = F.log_softmax(logits, -1)
    new_logp = log_all.gather(1, batch["actions"][:, None]).squeeze(1)
    advantages = batch["advantages"]
    advantages = (advantages - advantages.mean()) / advantages.std(unbiased=False).clamp_min(1e-6)
    surrogate, ratio = clipped_surrogate52(new_logp, batch["old_log_probs"], advantages, clip_ratio)
    policy_loss = -surrogate.mean()
    value_loss = 0.5 * F.mse_loss(values, batch["returns"])
    entropy = -(log_all.exp() * log_all).sum(-1).mean()
    total = policy_loss + value_coef * value_loss - entropy_coef * entropy
    return total, {"policy": policy_loss, "value": value_loss, "entropy": entropy, "ratio": ratio}

old52 = torch.zeros(2); new52 = torch.log(torch.tensor([1.5, 0.5])); signed_adv52 = torch.tensor([1.0, -1.0])
surrogate52, ratio52 = clipped_surrogate52(new52, old52, signed_adv52)
assert torch.allclose(ratio52, torch.tensor([1.5, 0.5]))
assert torch.allclose(surrogate52, torch.tensor([1.2, -0.8]), atol=1e-6)
loss_probe52, terms_probe52 = ppo_loss52(probe_model52, rollout_probe52)
assert loss_probe52.ndim == 0 and torch.isfinite(loss_probe52)
assert terms_probe52["ratio"].shape == (12,) and terms_probe52["entropy"] > 0
try:
    clipped_surrogate52(new52,old52,signed_adv52,float("inf")); raise AssertionError("infinite clip accepted")
except ValueError as error:
    assert str(error) == "surrogate_finite_clip_contract"
try:
    clipped_surrogate52(torch.tensor([0.]),torch.tensor([-1000.]),torch.tensor([-1.])); raise AssertionError("overflowing ratio accepted")
except ValueError as error:
    assert str(error) == "log_ratio_out_of_safe_range"

## 6. 受控训练协议

每轮先用冻结的当前策略收集 12 条 episode，再对同一 batch 做 4 次 PPO epoch。这里没有并行环境、minibatch shuffle 或 learning-rate schedule，以便观察核心机制。评估只用 `argmax`，报告四阶段动作准确率与总回报。

合成环境只有四个固定状态，达到 100% 是机制 smoke test，不代表探索、稀疏奖励、部分可观测或真实机器人控制上的泛化。

In [ ]:
def evaluate_policy52(model):
    env = PatternEnv52(); state = env.reset(); total=0.0; correct=0; actions=[]
    while True:
        action, _, _ = model.act(state[None, :], torch.Generator().manual_seed(0), deterministic=True)
        actions.append(int(action.item()))
        state, reward, terminated, truncated, info = env.step(actions[-1])
        total += reward; correct += int(info["correct"])
        if terminated or truncated:
            return total, correct / len(env.target), actions

torch.manual_seed(SEED52)
model52 = ActorCritic52().to(DEVICE52)
optimizer52 = torch.optim.Adam(model52.parameters(), lr=0.015)
rollout_generator52 = torch.Generator().manual_seed(SEED52 + 1)
initial_reward52, initial_accuracy52, _ = evaluate_policy52(model52)
loss_history52 = []
for update52 in range(35):
    batch52 = collect_rollouts52(model52, 12, rollout_generator52)
    for _ in range(4):
        optimizer52.zero_grad(set_to_none=True)
        loss52, pieces52 = ppo_loss52(model52, batch52)
        loss52.backward()
        torch.nn.utils.clip_grad_norm_(model52.parameters(), 1.0)
        optimizer52.step()
    loss_history52.append(float(loss52.detach()))
final_reward52, final_accuracy52, final_actions52 = evaluate_policy52(model52)
assert final_actions52 == list(PatternEnv52().target)
assert final_accuracy52 == 1.0 and final_reward52 == 4.0
assert final_reward52 >= initial_reward52 and all(math.isfinite(v) for v in loss_history52)
assert any(parameter.grad is not None and torch.isfinite(parameter.grad).all() for parameter in model52.parameters())
print({"initial_accuracy": initial_accuracy52, "final_accuracy": final_accuracy52, "actions": final_actions52})

## 7. 评估、失败模式与上线边界

常见错误包括：把 `truncated` 当 terminal；更新时重新计算 old log-prob；对负优势使用错误裁剪方向；rollout 与更新共享仍在变化的参数；只看训练 reward；部署时改变 observation/action 编码。

真实系统还需要多随机种子、置信区间、不同初态/扰动/OOD 场景、KL early stopping、value clipping、reward normalization、约束动作、安全 shield、离线策略评估、shadow/canary 和回滚。PPO 的 wall-clock 也常由环境采样而非 MLP FLOPs 主导。

In [ ]:
with torch.no_grad():
    eval_states52 = torch.eye(4)
    eval_logits52, eval_values52 = model52(eval_states52)
    eval_probs52 = eval_logits52.softmax(-1)
assert torch.equal(eval_probs52.argmax(-1), torch.tensor(PatternEnv52().target))
assert torch.all((eval_probs52 >= 0) & (eval_probs52 <= 1))
assert torch.isfinite(eval_values52).all()
swapped52 = eval_states52[[1, 0, 2, 3]]
with torch.no_grad(): swapped_logits52, _ = model52(swapped52)
assert torch.allclose(swapped_logits52, eval_logits52[[1, 0, 2, 3]], atol=1e-7)
try:
    PatternEnv52(target=(0, 2)); raise AssertionError("invalid action vocabulary accepted")
except ValueError as error:
    assert str(error) == "target_must_be_nonempty_binary_sequence"

## 8. 可信制品：发布者登记在 package 之外

package 绑定 state schema、环境/动作语义、固定评估状态、PPO/GAE 超参、bootstrap 选择、训练 seed 与完整 optimizer/梯度裁剪 recipe。loader 返回 `PublishedPolicy52`，服务只接收 stage 并在内部构造 one-hot，同时按发布 action map 返回动作名；裸的全零向量或越界动作不能越过边界。state 语义摘要逐 tensor 纳入 `key/dtype/shape/bytes`。

内部 `manifest_sha` 只能发现传输损坏，不能证明发布者身份。本例用只读 publisher registry 保存 `(artifact_id,version)->expected bundle digest`；即使攻击者替换模型并重算所有 package 内 hash，也无法改变 registry 的期望值。

In [ ]:
def semantic_state_digest52(state):
    digest = hashlib.sha256()
    for key, tensor in sorted(state.items()):
        value = tensor.detach().cpu().contiguous()
        digest.update(key.encode()); digest.update(str(value.dtype).encode())
        digest.update(canonical_json52(list(value.shape)).encode()); digest.update(value.numpy().tobytes())
    return digest.hexdigest()

manifest52 = {
    "artifact_id": "ppo-pattern-v1", "version": 1,
    "model_config": {"state_dim": 4, "hidden_dim": 24, "action_dim": 2},
    "environment": {"target": [0, 1, 1, 0], "reward_correct": 1.0, "reward_wrong": -0.25, "state_encoding": "stage_one_hot_v1"},
    "action_map": {"0": "choose_zero", "1": "choose_one"},
    "training_recipe": {"seed": SEED52, "optimizer": "Adam", "updates": 35, "episodes_per_update": 12, "rollout_max_steps": 4,
                        "epochs": 4, "lr": 0.015, "gamma": 0.9, "gae_lambda": 0.8, "clip_ratio": 0.2,
                        "value_coef": 0.5, "entropy_coef": 0.02, "grad_clip_norm": 1.0, "bootstrap_on_truncation": True},
    "eval_snapshot": {"states": torch.eye(4).tolist(), "gold_actions": [0, 1, 1, 0]},
}

def build_package52(model, manifest):
    buffer = io.BytesIO(); torch.save(model.state_dict(), buffer); state_bytes = buffer.getvalue()
    state = torch.load(io.BytesIO(state_bytes), map_location="cpu", weights_only=True)
    state_digest = semantic_state_digest52(state); bytes_sha = sha256_bytes52(state_bytes)
    manifest_sha = sha256_bytes52(canonical_json52(manifest).encode())
    bundle = sha256_bytes52(canonical_json52({"manifest_sha": manifest_sha, "state_digest": state_digest, "state_bytes_sha": bytes_sha}).encode())
    return {"manifest": copy.deepcopy(manifest), "manifest_sha": manifest_sha, "state_bytes": state_bytes,
            "state_digest": state_digest, "state_bytes_sha": bytes_sha, "bundle_digest": bundle}

package52 = build_package52(model52, manifest52)
PUBLISHER_REGISTRY52 = MappingProxyType({("ppo-pattern-v1", 1): package52["bundle_digest"]})

class PublishedPolicy52:
    def __init__(self, model, manifest):
        self._model = model
        self.state_dim = int(manifest["model_config"]["state_dim"])
        self.target = tuple(manifest["environment"]["target"])
        self.action_map = MappingProxyType({int(key): value for key,value in manifest["action_map"].items()})

    def __call__(self, states):
        return self._model(states)

    @torch.no_grad()
    def act_stage(self, stage):
        if isinstance(stage,bool) or not isinstance(stage,(int,np.integer)) or not 0 <= int(stage) < self.state_dim:
            raise ValueError("stage_out_of_range")
        state = F.one_hot(torch.tensor([int(stage)]), self.state_dim).float()
        action, log_prob, value = self._model.act(state, torch.Generator().manual_seed(0), deterministic=True)
        action_id = int(action.item())
        return {"action_id": action_id, "action_name": self.action_map[action_id], "log_prob": float(log_prob), "value": float(value)}

def load_policy52(package):
    manifest = package["manifest"]; key = (manifest.get("artifact_id"), manifest.get("version"))
    if PUBLISHER_REGISTRY52.get(key) != package.get("bundle_digest"):
        raise RuntimeError("publisher_registry_rejected_bundle")
    if manifest != manifest52 or sha256_bytes52(canonical_json52(manifest).encode()) != package["manifest_sha"]:
        raise RuntimeError("manifest_contract_mismatch")
    if sha256_bytes52(package["state_bytes"]) != package["state_bytes_sha"]:
        raise RuntimeError("state_bytes_mismatch")
    state = torch.load(io.BytesIO(package["state_bytes"]), map_location="cpu", weights_only=True)
    if semantic_state_digest52(state) != package["state_digest"]:
        raise RuntimeError("state_semantic_digest_mismatch")
    expected_bundle = sha256_bytes52(canonical_json52({"manifest_sha": package["manifest_sha"], "state_digest": package["state_digest"], "state_bytes_sha": package["state_bytes_sha"]}).encode())
    if expected_bundle != package["bundle_digest"]:
        raise RuntimeError("bundle_digest_mismatch")
    restored = ActorCritic52(**manifest["model_config"]); restored.load_state_dict(state); restored.eval()
    return PublishedPolicy52(restored, manifest)

restored52 = load_policy52(package52)
with torch.no_grad():
    assert torch.allclose(restored52(torch.eye(4))[0], model52(torch.eye(4))[0])
served_stage52 = restored52.act_stage(1)
assert served_stage52["action_id"] == 1 and served_stage52["action_name"] == "choose_one"
assert isinstance(restored52.action_map, MappingProxyType) and restored52.target == (0,1,1,0)
try:
    restored52.act_stage(4); raise AssertionError("invalid stage accepted")
except ValueError as error:
    assert str(error) == "stage_out_of_range"
forged_model52 = ActorCritic52()
with torch.no_grad():
    for parameter in forged_model52.parameters(): parameter.zero_()
forged_package52 = build_package52(forged_model52, manifest52)
try:
    load_policy52(forged_package52); raise AssertionError("fully re-signed forged model accepted")
except RuntimeError as error:
    assert str(error) == "publisher_registry_rejected_bundle"
assert isinstance(PUBLISHER_REGISTRY52, MappingProxyType)

## 9. 面试复盘与原始来源

回答 PPO 工程题时，先讲 rollout/version 合同，再写 GAE 的两个 mask、ratio 与正负优势裁剪，最后说明多 epoch 更新、KL/entropy/value、离线评估和发布安全。只背“clip 可以稳定训练”不足以解释实现是否正确。

- Schulman et al., [Proximal Policy Optimization Algorithms](https://arxiv.org/abs/1707.06347), 2017。
- Schulman et al., [High-Dimensional Continuous Control Using Generalized Advantage Estimation](https://arxiv.org/abs/1506.02438), ICLR 2016。
- Sutton & Barto, [Reinforcement Learning: An Introduction](http://incompleteideas.net/book/the-book-2nd.html)，MDP 与 bootstrap 背景。

本例是离散、全可观测、固定四步环境的机制测试，没有证明真实策略的样本效率、安全性或泛化。